# Bulwark: train the N4 net on the GPU (Phase 3)

Trains **N4** (8 king buckets x 768 -> 1024, 8 output buckets, dual head) on the 1.5B-position Lc0-label dataset
built by the five `bulwark_data_lc0_s*` notebooks. Then it exports the net, checks the engine reproduces it
(`verifynet`), and scores it on the held-out test split.

**Settings (right-hand panel):** Accelerator **GPU** (P100 or T4; one GPU is used), Internet **On** (only used by
`pip` if something is missing; data and code come from inputs).

**Inputs (Add Input → Your Work):**
1. the `bulwark` dataset (with `bulwark.zip`, the latest version);
2. the **outputs of all five Lc0 data notebooks** (`bulwark_data_lc0_s0` ... `_s4`; Notebook Output Files).

**Run it:** Save Version → **Save & Run All (Commit)**.

**Checkpointing:** every run saves a checkpoint every 20 minutes into this notebook's output, and the notebook stops
cleanly at 11 h (Kaggle GPU sessions end at 12 h). If the last cell says **NOT DONE**: Add Input → Your Work →
the output of the previous version of *this* notebook, then Save & Run All again. Finished runs are skipped and
the interrupted run continues from its last checkpoint. Each session uses part of your 30 GPU-hours per week.

In [ ]:
# ---- Parameters -------------------------------------------------------------
STOP_AFTER_H = 11.0      # stop with a checkpoint after this many hours (session limit 12 h)
SMOKE = False            # True: 20 steps per run, to test the setup in a few minutes
RUNS = "n4"              # which queue runs to train (scripts/gpu_queue.sh)

import os, time, glob, json, subprocess, shutil
START = time.time()
WORK = "/kaggle/working"     # saved when the version finishes: ckpt/, nets/, logs/
TMP = "/kaggle/tmp"          # code and the assembled data folder (not saved)
os.makedirs(TMP, exist_ok=True)
def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)
sh("nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv")
print("CPUs:", os.cpu_count())

In [ ]:
# ---- Code: clone from the bulwark dataset and build the engine (for verifynet) ----
bundles = glob.glob("/kaggle/input/**/bulwark.bundle", recursive=True)
if not bundles:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        sh(f"unzip -o -q '{z}' -d {TMP}/bundle")
    bundles = glob.glob(f"{TMP}/bundle/**/bulwark.bundle", recursive=True)
assert bundles, "add the bulwark dataset (bulwark.zip) as an input"
CODE = f"{TMP}/bulwark"
if not os.path.isdir(CODE):
    sh(f"git clone -q '{bundles[0]}' {CODE}")
sh(f"cd {CODE} && git log --oneline -1")
sh(f"cd {CODE} && cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j{os.cpu_count()} --target bulwark > /dev/null")
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "set Accelerator to GPU"

In [ ]:
# ---- Data: the five data notebooks' training shards -> one data/training folder ----
# train/ is linked (read-only inputs, 1.7 GB); val/ and test/ are copied so positions that also occur in train
# can be removed from them (the same step the server's dataset went through).
found = {}
for f in glob.glob("/kaggle/input/**/bulwark_data/s*/training/*/s*-*.bin", recursive=True):
    split, name = f.split("/")[-2], f.split("/")[-1]
    found.setdefault((split, name), f)
sessions = sorted({n.split("-")[0] for _, n in found})
print("sessions:", sessions)
assert sessions == ["s0", "s1", "s2", "s3", "s4"], "add the outputs of all five data notebooks as inputs"
DATA = f"{CODE}/data/training"
shutil.rmtree(DATA, ignore_errors=True)
for split in ("train", "val", "test"):
    os.makedirs(f"{DATA}/{split}")
for (split, name), f in sorted(found.items()):
    if split == "train":
        os.symlink(f, f"{DATA}/{split}/{name}")
    else:
        shutil.copyfile(f, f"{DATA}/{split}/{name}")
sh(f"cd {CODE} && python3 -m pipeline.check_splits --training {DATA} --drop-overlap | grep -E 'records|overlap_with_train'")

In [ ]:
# ---- Restore: continue from an earlier version's output, if one is attached ----
def progress(ck):
    runs = glob.glob(f"{ck}/run-*")
    done = sum(os.path.exists(f"{r}/DONE") for r in runs)
    steps = 0
    for r in runs:
        for p in glob.glob(f"{r}/step-*.pt"):
            steps = max(steps, int(p.rsplit("-", 1)[1].split(".")[0]))
    return (done, steps)
CK = "ckpt_smoke" if SMOKE else "ckpt"
prev = [d for d in glob.glob(f"/kaggle/input/**/{CK}", recursive=True) if glob.glob(f"{d}/run-*")]
if prev:
    best = max(prev, key=progress)
    root = os.path.dirname(best)
    print("restoring from", root, "progress (runs done, max step):", progress(best))
    for sub in (("ckpt_smoke", "nets_smoke", "logs_smoke") if SMOKE else ("ckpt", "nets", "logs")):
        if os.path.isdir(f"{root}/{sub}"):
            shutil.copytree(f"{root}/{sub}", f"{WORK}/{sub}", dirs_exist_ok=True)
else:
    print("no earlier output attached: starting fresh")

In [ ]:
# ---- Train: the same queue as the server (one GPU, checkpoints every 20 min, stops at STOP_AFTER_H) ----
env = dict(os.environ,
           DATA=DATA, CKPT=f"{WORK}/ckpt", NETS=f"{WORK}/nets", LOGS=f"{WORK}/logs",
           GPU_MEM_FRACTION="0.9", TORCH_THREADS="2", LOADER_WORKERS=str(max(1, os.cpu_count() - 1)),
           TRAIN_STOP_AT=str(int(START + STOP_AFTER_H * 3600)),
           RUNS=RUNS,
           PYTHONUSERBASE=os.path.expanduser("~/.local"))   # keep Kaggle's own user packages visible
if SMOKE:
    env.update(SMOKE="1", CKPT=f"{WORK}/ckpt_smoke", NETS=f"{WORK}/nets_smoke", LOGS=f"{WORK}/logs_smoke")
p = subprocess.Popen(["bash", "scripts/gpu_queue.sh"], cwd=CODE, env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
rc = p.wait()
print("queue exit code", rc)

In [ ]:
# ---- Report ----
ck, logs = (f"{WORK}/ckpt_smoke", f"{WORK}/logs_smoke") if SMOKE else (f"{WORK}/ckpt", f"{WORK}/logs")
runs = RUNS.split()
for r in runs:
    m = f"{ck}/run-{r}/metrics.jsonl"
    last = json.loads(open(m).read().splitlines()[-1]) if os.path.exists(m) else {}
    vals = [json.loads(l) for l in open(m)] if os.path.exists(m) else []
    vl = [v["val_logloss"] for v in vals if "val_logloss" in v]
    print(f"{r:10s} {'DONE' if os.path.exists(f'{ck}/run-{r}/DONE') else 'not done':9s} step {last.get('step', 0):>7}  "
          f"last val_logloss {vl[-1] if vl else '-'}")
for f in sorted(glob.glob(f"{logs}/*.log")):
    for line in open(f):
        if line.startswith("verifynet"):
            print(os.path.basename(f), line.strip())
if os.path.exists(f"{logs}/evaluate.json"):
    print(open(f"{logs}/evaluate.json").read())
done = all(os.path.exists(f"{ck}/run-{r}/DONE") for r in runs)
print("\nDONE: nets are in", f"{WORK}/{'nets_smoke' if SMOKE else 'nets'}" if done else "")
if not done:
    print("NOT DONE: add this version's output as an input and Save & Run All again (it resumes).")